# 01 — Exploratory Data Analysis

EDA on the ingested CERT-schema communication dataset (real if present in `data/raw/`, else the synthetic generator's output — see `src/data/synthetic_cert_data.py`). Applies the EDA workflow from the NASSCOM course: shape/structure checks, missingness, distributions, time-based patterns, and per-user activity profiling — before any feature engineering happens.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.ingest import load_email_data, load_insider_labels

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## Load data

In [ ]:
df = load_email_data()
print(df.shape)
df.head()

## Structure & missingness

In [ ]:
df.info()
print('\nMissing values per column:')
print(df.isna().sum())

## Message volume over time

In [ ]:
daily_counts = df.set_index('date').resample('D').size()

fig, ax = plt.subplots(figsize=(11, 4))
daily_counts.plot(ax=ax, color='#0f62fe')
ax.set_title('Messages per day')
ax.set_ylabel('message count')
plt.tight_layout()
plt.show()

## Per-user activity distribution

In [ ]:
user_counts = df['user'].value_counts()

fig, ax = plt.subplots(figsize=(11, 4))
sns.histplot(user_counts, bins=25, color='#0f62fe', ax=ax)
ax.set_title('Distribution of message counts per user')
ax.set_xlabel('messages sent')
plt.tight_layout()
plt.show()

user_counts.describe()

## Message length distribution

In [ ]:
df['content_length'] = df['content'].str.len()

fig, ax = plt.subplots(figsize=(11, 4))
sns.histplot(df['content_length'], bins=40, color='#007d79', ax=ax)
ax.set_title('Message length distribution (characters)')
plt.tight_layout()
plt.show()

## Ground-truth label balance (synthetic run only)

Only available when `insider_labels.csv` exists (always true for the synthetic generator; supply your own if using real CERT data + real answer keys).

In [ ]:
labels = load_insider_labels()
if labels is not None:
    print(labels['is_malicious_insider'].value_counts())
    labels['is_malicious_insider'].value_counts().plot(kind='bar', color=['#0f62fe', '#da1e28'])
    plt.title('Insider label balance')
    plt.xticks([0, 1], ['normal', 'insider'], rotation=0)
    plt.show()
else:
    print('No label file found.')

## Takeaways

- Message volume and per-user activity should look roughly stable/periodic for normal users — the feature/baseline pipeline (Part 1) is built to detect *deviation* from exactly this kind of stable pattern, so it's worth confirming the baseline period actually looks stable before trusting drift scores downstream.
- Label imbalance (very few insiders vs. many normal users) is expected and is why the supervised model uses `class_weight='balanced'` / `scale_pos_weight` (see `src/models/supervised/train.py`), and why the unsupervised models (IsolationForest/DBSCAN) matter — they don't need balanced labels at all.